In [3]:
%load_ext autoreload
%autoreload 1

import os, sys, random
import importlib
import torchaudio
from pathlib import Path

import torch
import torchbend
torchbend.set_output('notebook')
from torchbend.interfaces.descript_interface import BendedDescriptAudioCodec

import ipywidgets as widgets
from IPython.display import Audio, display

torch.set_grad_enabled(False)

def audio_grid(items, columns=3):
    # items: list of (label, numpy_array_or_tensor, sr)
    cells = []
    for label, wav, sr in items:
        out = widgets.Output()
        with out:
            display(widgets.Label(str(label)))
            display(Audio(wav.numpy(), rate=sr))
        cells.append(out)

    return widgets.GridBox(
        cells,
        layout=widgets.Layout(grid_template_columns=f"repeat({columns}, 1fr)")
    )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
from torchbend.utils import load_audio

audio_dir = "data/test"

audio_files = load_audio(audio_dir)
trace_n_samples = audio_files.max_size()

device = torch.device('cuda')
interface = BendedDescriptAudioCodec(model_type="44khz", device=torch.device('cuda'), trace_n_samples=trace_n_samples, trace_n_batches=1)

/root/torchbend/miniconda3/envs/torchbend/lib/python3.11/site-packages/audiotools/ml/layers/base.py:172: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_dict = torch.loa

In [5]:
x = audio_files.as_list(channels=1, sr=interface.sample_rate, size=trace_n_samples)
z, codes, latents, _, _ = interface.encode(x[0][None])

## Messing up directly with decoder's input

In [16]:
import torchaudio
from IPython.display import Audio
print(z.shape, codes.shape, latents.shape)

def mask_dim(z, prob):
    z = z * torch.bernoulli(torch.full((z.shape[-2],), prob)).reshape(1, -1, 1).to(z)
    return z
def mask_time_dim(z, prob):
    z = z * torch.bernoulli(torch.full((z.shape[-2], z.shape[-1]), prob)).reshape(1, z.size(-2), z.size(-1)).to(z)
    return z
def mask_extreme_dims(z, prob, largest=True):
    # rank channels by mean absolute activation over time, zero the top or bottom prob%
    activation = z.abs().mean(-1).mean(0)          # [D]
    k = max(1, int(prob * z.shape[-2]))
    indices = activation.topk(k, largest=largest).indices
    mask = torch.ones(z.shape[-2], device=z.device, dtype=z.dtype)
    mask[indices] = 0
    return z * mask.reshape(1,-1,1)

out_dir = Path("outs/dac/z_mask")
os.makedirs(out_dir, exist_ok=True)

probs = torch.linspace(0.1, 1, 10).tolist()
out_files = []
for p in probs:
    torch.cuda.empty_cache()
    z_bended = mask_extreme_dims(z, p)
    out_bended = interface.decode(z_bended).cpu()[0]
    out_path = out_dir / f"mask_prob={p:.2f}.wav"
    out_files.append((out_path, out_bended, interface.sample_rate))
    torchaudio.save(out_dir / f"mask_prob={p:.2f}.wav", out_bended, interface.sample_rate)
    del z_bended, out_bended

audio_grid(out_files)


torch.Size([1, 1024, 500]) torch.Size([1, 9, 500]) torch.Size([1, 72, 500])


GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Ou…

In [20]:
import torchaudio
from IPython.display import Audio
print(z.shape, codes.shape, latents.shape)

def affine_transform(z, scale, bias):
    z = scale * z + bias
    return z

out_dir = Path("outs/dac/z_scale")
os.makedirs(out_dir, exist_ok=True)

scale = [1., 2., 4.]
bias = [-4., -3., 3., 4.]
out_files = []
for s in scale:
    for b in bias:
        torch.cuda.empty_cache()
        z_bended = affine_transform(z, s, b)
        out_bended = interface.decode(z_bended).cpu()[0]
        out_path = out_dir / f"affine_scale={s:.2f}_bias={b:.2f}.wav"
        out_files.append((out_path, out_bended, interface.sample_rate))
        torchaudio.save(out_path, out_bended, interface.sample_rate)
        del z_bended, out_bended

audio_grid(out_files)


torch.Size([1, 1024, 500]) torch.Size([1, 9, 500]) torch.Size([1, 72, 500])


GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Ou…

In [ ]:
import random
import torchaudio
from IPython.display import Audio
print(z.shape, codes.shape, latents.shape)

def permute_dims(z, seed=None):
    gen = torch.Generator(device=z.device)
    if seed is not None:
        gen.manual_seed(seed)
    perm = torch.randperm(z.shape[-2], device=z.device, generator=gen).to(z.device)
    return z[:, perm, :]

out_dir = Path("outs/dac/z_permute")
os.makedirs(out_dir, exist_ok=True)

out_files = []

seeds = [random.randrange(0, 10000) for _ in range(6)]

for s in seeds:
    torch.cuda.empty_cache()
    z_bended = permute_dims(z, s)
    out_bended = interface.decode(z_bended).cpu()[0]
    out_path = out_dir / f"z_seed={s}.wav"
    out_files.append((out_path, out_bended, interface.sample_rate))
    torchaudio.save(out_path, out_bended, interface.sample_rate)
    del z_bended, out_bended

audio_grid(out_files)


torch.Size([1, 1024, 500]) torch.Size([1, 9, 500]) torch.Size([1, 72, 500])


GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Ou…

In [ ]:
# z.shape, codes.shape, latents.shape
interface.print_activations(fn="encode")

# direct latent output : convolution_29
# projections used by quantization : quantizer_quantizers_8_out_proj_weight_v
# quantizer codebooks weights : quantizer_quantizers_0_codebook_weight

name,op,target,shape,args,kwargs
audio_data,placeholder,audio_data,"(1, 1, s0)",(),{}
n_quantizers,placeholder,n_quantizers,None,"(None,)",{}
encoder_block_0_weight_v,get_attr,encoder.block.0.weight_v,"(64, 1, 7)",(),{}
encoder_block_0_weight_g,get_attr,encoder.block.0.weight_g,"(64, 1, 1)",(),{}
_weight_norm_interface,call_function,aten._weight_norm_interface.default,None,"(encoder_block_0_weight_v, encoder_block_0_weight_g)",{}
getitem,call_function,<built-in function getitem>,"(64, 1, 7)","(_weight_norm_interface, 0)",{}
getitem_1,call_function,<built-in function getitem>,"(64, 1, 1)","(_weight_norm_interface, 1)",{}
encoder_block_0_bias,get_attr,encoder.block.0.bias,"(64,)",(),{}
convolution,call_function,aten.convolution.default,"(1, 64, s0)","(audio_data, getitem, encoder_block_0_bias, [1], [3], [1], False, [0], 1)",{}
view,call_function,aten.view.default,"(1, 64, s0)","(convolution, [1, 64, -1])",{}


"----------------------------------------  -------------  ----------------------------------------  --------------------  -----------------------------------------------------------------------------------------------------------------------------------------------------  -------------------------------------------------------------\naudio_data                                placeholder    audio_data                                (1, 1, s0)            ()                                                                                                                                                     {}\nn_quantizers                              placeholder    n_quantizers                                                    (None,)                                                                                                                                                {}\nencoder_block_0_weight_v                  get_attr       encoder.block.0.weight_v                  (64, 1, 7)   

## Messing with codes

In [3]:
from torchbend.utils import load_audio

audio_dir = "data/test"

audio_files = load_audio(audio_dir)
trace_n_samples = audio_files.max_size()

x = audio_files.as_list(channels=1, sr=interface.sample_rate, size=trace_n_samples)
z_1, codes_1, latents_1, _, _ = interface.encode(x[0][None])
z_2, codes_2, latents_2, _, _ = interface.encode(x[1][None])


In [42]:
import random
import torchaudio
from IPython.display import Audio
print(z.shape, codes.shape, latents.shape)


def mix_dims(z1, z2, idx):
    out = torch.empty_like(z1)
    out[:, :idx, :] = z1[:, :idx, :]
    out[:, idx:, :] = z2[:, idx:, :]
    return out

out_dir = Path("outs/dac/code_play")
os.makedirs(out_dir, exist_ok=True)

out_files = []
dec_idx = [1,2,3]

for i in dec_idx:
    torch.cuda.empty_cache()
    codes_bended = mix_dims(codes_1, codes_2, i)
    z_bended, _, _= interface.model.quantizer.from_codes(codes_bended)
    out_bended = interface.decode(z_bended).cpu()[0]
    out_path = out_dir / f"codes_dim={i}.wav"
    out_files.append((out_path, out_bended, interface.sample_rate))
    torchaudio.save(out_path, out_bended, interface.sample_rate)
    del codes_bended, z_bended, out_bended

audio_grid(out_files)

torch.Size([1, 1024, 500]) torch.Size([1, 9, 500]) torch.Size([1, 72, 500])


GridBox(children=(Output(), Output(), Output()), layout=Layout(grid_template_columns='repeat(3, 1fr)'))

In [50]:
import random
import torchaudio
from IPython.display import Audio
print(z.shape, codes.shape, latents.shape)

def blend_dims(z1, z2, seed=None):
    gen = torch.Generator(device=z1.device)
    if seed is not None:
        gen.manual_seed(seed)
    mask = torch.randint(0, 2, (z1.shape[-2],), device=z1.device, generator=gen)
    mask = mask.reshape(1, -1, 1).to(z1.dtype)
    return z1 * mask + z2 * (1 - mask)

out_dir = Path("outs/dac/code_play")
os.makedirs(out_dir, exist_ok=True)

out_files = []
seeds = [random.randrange(0, 10000) for _ in range(6)]

for i in seeds:
    torch.cuda.empty_cache()
    codes_bended = blend_dims(codes_1, codes_2, i)
    z_bended, _, _= interface.model.quantizer.from_codes(codes_bended)
    out_bended = interface.decode(z_bended).cpu()[0]
    out_path = out_dir / f"codes_blend_seed={i}.wav"
    out_files.append((out_path, out_bended, interface.sample_rate))
    torchaudio.save(out_path, out_bended, interface.sample_rate)
    del codes_bended, z_bended, out_bended

audio_grid(out_files)

torch.Size([1, 1024, 500]) torch.Size([1, 9, 500]) torch.Size([1, 72, 500])


GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output()), layout=Layout(grid_template_col…

In [ ]:
import random
import torchaudio
from IPython.display import Audio
print(z.shape, codes.shape, latents.shape)

def blend_dims(z1, z2, seed=None):
    gen = torch.Generator(device=z1.device)
    if seed is not None:
        gen.manual_seed(seed)
    mask = torch.randint(0, 2, (z1.shape[-2],), device=z1.device, generator=gen)
    mask = mask.reshape(1, -1, 1).to(z1.dtype)
    return z1 * mask + z2 * (1 - mask)

out_dir = Path("outs/dac/code_play")
os.makedirs(out_dir, exist_ok=True)

out_files = []
seeds = [random.randrange(0, 10000) for _ in range(6)]

for i in seeds:
    torch.cuda.empty_cache()
    z_bended = blend_dims(z_1, z_2, i)
    out_bended = interface.decode(z_bended).cpu()[0]
    out_path = out_dir / f"codes_blend_seed={i}.wav"
    out_files.append((out_path, out_bended, interface.sample_rate))
    torchaudio.save(out_path, out_bended, interface.sample_rate)
    del z_bended, out_bended

audio_grid(out_files)

torch.Size([1, 1024, 500]) torch.Size([1, 9, 500]) torch.Size([1, 72, 500])
tensor([[[0.],
         [1.],
         [1.],
         ...,
         [1.],
         [1.],
         [1.]]], device='cuda:0')


tensor([[[0.],
         [0.],
         [0.],
         ...,
         [1.],
         [0.],
         [0.]]], device='cuda:0')
tensor([[[0.],
         [1.],
         [1.],
         ...,
         [1.],
         [0.],
         [0.]]], device='cuda:0')
tensor([[[1.],
         [1.],
         [0.],
         ...,
         [1.],
         [0.],
         [1.]]], device='cuda:0')
tensor([[[1.],
         [1.],
         [1.],
         ...,
         [1.],
         [1.],
         [1.]]], device='cuda:0')
tensor([[[1.],
         [1.],
         [0.],
         ...,
         [1.],
         [1.],
         [1.]]], device='cuda:0')


GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output()), layout=Layout(grid_template_col…

In [51]:
import random
import torchaudio
from IPython.display import Audio
print(z.shape, codes.shape, latents.shape)

def sum_max_dims(z1, z2, prob):
    k = max(1, int(prob * z1.shape[-2]))

    a1 = z1.abs().mean(-1).mean(0)  # [D]
    a2 = z2.abs().mean(-1).mean(0)  # [D]

    mask1 = torch.zeros(z1.shape[-2], device=z1.device, dtype=z1.dtype)
    mask1[a1.topk(k).indices] = 1

    mask2 = torch.zeros(z2.shape[-2], device=z2.device, dtype=z2.dtype)
    mask2[a2.topk(k).indices] = 1

    return z1 * mask1.reshape(1, -1, 1) + z2 * mask2.reshape(1, -1, 1)

out_dir = Path("outs/dac/code_play")
os.makedirs(out_dir, exist_ok=True)

out_files = []
probs=[0.1, 0.2, 0.5, 0.8]

for i in probs:
    torch.cuda.empty_cache()
    z_bended = sum_max_dims(z_1, z_2, i)
    out_bended = interface.decode(z_bended).cpu()[0]
    out_path = out_dir / f"codes_blend_seed={i}.wav"
    out_files.append((out_path, out_bended, interface.sample_rate))
    torchaudio.save(out_path, out_bended, interface.sample_rate)
    del z_bended, out_bended

audio_grid(out_files)

torch.Size([1, 1024, 500]) torch.Size([1, 9, 500]) torch.Size([1, 72, 500])


GridBox(children=(Output(), Output(), Output(), Output()), layout=Layout(grid_template_columns='repeat(3, 1fr)…

In [53]:
import random
import torchaudio
from IPython.display import Audio

def corrupt_codes(codes, prob, n_codes=1024, seed=None):
    gen = torch.Generator(device=codes.device)
    if seed is not None:
        gen.manual_seed(seed)
    mask = torch.bernoulli(torch.full(codes.shape, prob, device=codes.device, dtype=torch.float), generator=gen).bool()
    random_codes = torch.randint(0, n_codes, codes.shape, device=codes.device, generator=gen)
    return torch.where(mask, random_codes, codes)

out_dir = Path("outs/dac/code_play")
os.makedirs(out_dir, exist_ok=True)

out_files = []
probs=[0.1, 0.2, 0.5, 0.8]

for i in probs:
    torch.cuda.empty_cache()
    codes_bended = corrupt_codes(codes_1, i)
    z_bended, _, _= interface.model.quantizer.from_codes(codes_bended)
    out_bended = interface.decode(z_bended).cpu()[0]
    out_path = out_dir / f"codes_blend_seed={i}.wav"
    out_files.append((out_path, out_bended, interface.sample_rate))
    torchaudio.save(out_path, out_bended, interface.sample_rate)
    del codes_bended, z_bended, out_bended

audio_grid(out_files)

GridBox(children=(Output(), Output(), Output(), Output()), layout=Layout(grid_template_columns='repeat(3, 1fr)…

In [34]:
import torchaudio
import random


bended_q = torchbend.BendedModule(interface.model.quantizer)
bended_q.trace(fn="from_codes", codes=codes_1)

out_dir = Path("outs/dac/code_play")
os.makedirs(out_dir, exist_ok=True)


def noise_dims(x, prob=None, std=1.0, seed=None):
    gen = torch.Generator(device=x.device)
    if seed is not None:
        gen.manual_seed(int(seed))
    mask = torch.bernoulli(torch.full((x.shape[-2],), prob, device=x.device), generator=gen)
    noise = torch.randn(x.shape, device=x.device, generator=gen) * std
    return x + noise * mask.reshape(-1, 1)

def permute_dims(x, seed = None):
    gen = torch.Generator(device=x.device)
    if seed is not None:
        gen.manual_seed(int(seed.item()))
    perm = torch.randperm(x.shape[-2], device=x.device, generator=gen)
    return x[perm, :]


out_files = []
# dims = [1, 2, 4, 8]
dims = [0,]
probs=[0.5, 0.8]
seeds = [random.randrange(1000) for _ in range(8)]

target = r"?quantizers.[1-2].codebook.weight"

# for p in probs:
#     torch.cuda.empty_cache()

#     bended_q.reset()
#     cb = torchbend.Lambda(noise_dims, prob=p)
#     bended_q.bend(cb, target, fn="from_codes")

#     z_bended, _, _ = bended_q.from_codes(codes_2)
#     out_bended = interface.decode(z_bended).cpu()[0]
#     out_path = out_dir / f"codes_blend_prob={i}.wav"
#     out_files.append((out_path, out_bended, interface.sample_rate))
#     torchaudio.save(out_path, out_bended, interface.sample_rate)
#     del z_bended, out_bended
    
for d in dims: 
    for s in seeds:
        torch.cuda.empty_cache()

        bended_q.reset()
        target = f"?quantizers.{d}.codebook.weight"
        # cb = torchbend.Lambda(noise_dims, prob=i)
        cb = torchbend.Lambda(permute_dims, seed=s)
        bended_q.bend(cb, target, fn="from_codes")

        z_bended, _, _ = bended_q.from_codes(codes_2)
        out_bended = interface.decode(z_bended).cpu()[0]
        out_path = out_dir / f"codes_blend_dim={d}_seed={i}.wav"
        out_files.append((out_path, out_bended, interface.sample_rate))
        torchaudio.save(out_path, out_bended, interface.sample_rate)
        del z_bended, out_bended

print(bended_q.bended_weights)
audio_grid(out_files)

{'quantizers.0.codebook.weight': [Lambda(fn=permute_dims)]}


GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output()), layout=Layo…

### Play with projections

In [ ]:
import torchaudio
import random
#quantizers.1.in_proj.weight_v	: (8, 1024, 1)	
# bended_q.print_weights()
bended_q = torchbend.BendedModule(interface.model.quantizer)
bended_q.trace(fn="from_codes", codes=codes_1)

out_dir = Path("outs/dac/code_play")
os.makedirs(out_dir, exist_ok=True)


def noise_dims(x, prob=None, std=0.5, seed=None):
    gen = torch.Generator(device=x.device)
    if seed is not None:
        gen.manual_seed(int(seed))
    mask = torch.bernoulli(torch.full((x.shape[1],), prob, device=x.device), generator=gen)
    noise = torch.randn(x.shape, device=x.device, generator=gen) * std
    return x + noise * mask.reshape(1, -1, 1)
    # return torch.zeros_like(x)

def permute_dims(x, seed = None):
    gen = torch.Generator(device=x.device)
    if seed is not None:
        gen.manual_seed(int(seed.item()))
    perm = torch.randperm(x.shape[1], device=x.device, generator=gen)
    return x[:, perm, :]


out_files = []
probs=[0.3, 0.5]
seeds = [random.randrange(1000) for _ in range(8)]

target = r"?quantizers.\d.out_proj.weight_v"

# for p in probs:
#     torch.cuda.empty_cache()

#     bended_q.reset()
#     cb = torchbend.Lambda(noise_dims, prob=p)
#     bended_q.bend(cb, target, fn="from_codes")

#     z_bended, _, _ = bended_q.from_codes(codes_2)
#     out_bended = interface.decode(z_bended).cpu()[0]
#     out_path = out_dir / f"proj_blend_prob={p}.wav"
#     out_files.append((out_path, out_bended, interface.sample_rate))
#     torchaudio.save(out_path, out_bended, interface.sample_rate)
#     del z_bended, out_bended
    

for i in dim:
    for s in seeds:
        torch.cuda.empty_cache()

        bended_q.reset()
        # cb = torchbend.Lambda(noise_dims, prob=i)
        cb = torchbend.Lambda(permute_dims, seed=s)
        bended_q.bend(cb, target, fn="from_codes")

        z_bended, _, _ = bended_q.from_codes(codes_2)
        out_bended = interface.decode(z_bended).cpu()[0]
        out_path = out_dir / f"proj_blend_seed={s}.wav"
        out_files.append((out_path, out_bended, interface.sample_rate))
        torchaudio.save(out_path, out_bended, interface.sample_rate)
        del z_bended, out_bended

print(bended_q.bended_weights)
audio_grid(out_files)

{'quantizers.0.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.1.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.2.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.3.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.4.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.5.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.6.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.7.out_proj.weight_v': [Lambda(fn=permute_dims)], 'quantizers.8.out_proj.weight_v': [Lambda(fn=permute_dims)]}


GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output()), layout=Layo…

## Messing with the encoder

In [85]:
acts = interface.activations(r"?add_\d+", fn="encode")
audio_acts = []
for name, prop in acts.items(): 
    if len(prop.shape) < 1: continue
    if isinstance(prop.shape[-1], torch.SymInt):
        audio_acts.append(name)
print("number of valid activations : ", len(audio_acts))

number of valid activations :  59


In [63]:
target_acts = [3 * i for i in [5, 8, 10, 15]]

mask = torchbend.BendingParameter(name="prob", value=1.0)
cb = torchbend.ThresholdActivation(threshold=mask, dim=-2)

probs = [0.01, 0.1, 0.5, 0.8]
audio  = x[0][None]


out_dir = Path("outs/dac/encode_play")
os.makedirs(out_dir, exist_ok=True)
out_files = []

for t in target_acts: 
    interface.reset()
    interface.bend(cb, audio_acts[t], fn="encode")
    for p in probs:
        torch.cuda.empty_cache()
        mask.set_value(p)
        z, _, _, _, _ = interface.encode(audio)
        out_bended = interface.decode(z).cpu()[0]

        out_path = out_dir / f"encode_{t}_p={p:.2f}.wav"
        out_files.append((out_path, out_bended, interface.sample_rate))
        torchaudio.save(out_path, out_bended, interface.sample_rate)
        del out_bended, z
    
audio_grid(out_files)
        



GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Ou…

In [84]:
from IPython.display import Audio
target_acts = [3 * i for i in [2, 5, 8, 10, 15, 18]]

x = audio_files.as_list(channels=1, sr=interface.sample_rate, size=44100 * 4)

def max_callback(act_1, act_2): 
    return torch.where(act_1 > act_2, act_1, act_2)

def min_callback(act_1, act_2):
    return torch.where(act_1 < act_2, act_1, act_2)

def sub_callback(act_1, act_2):
    perm = torch.randperm(act_2.size(-2)).to(act_2.device)
    return act_2[:, perm, :]

def stochastic_mix(z1, z2, prob=0.8, seed=64):
    gen = torch.Generator(device=z1.device)
    if seed is not None:
        gen.manual_seed(seed)
    mask = torch.bernoulli(torch.full((z1.shape[-2],), prob, device=z1.device), generator=gen)
    mask = mask.reshape(1, -1, 1).to(z1)
    return z1 * mask + z2 * (1 - mask)

device = torch.device('cuda')
target_act = audio_acts[target_acts[2]+2]
act_1 = interface.get_activations(target_act, audio_data=x[0][None].to(device), fn="encode")[target_act]
act_2 = interface.get_activations(target_act, audio_data=x[1][None].to(device), fn="encode")[target_act]


act_mixed = {"audio_data":x[0][None].to(device)*1, target_act: max_callback(act_1, act_2)}
z, _, _, _, _ = interface.from_activations(target_act, **act_mixed, fn="encode")

out = interface.decode(z).cpu()[0]

out_dir = Path("outs/dac/enc_interp")
os.makedirs(out_dir, exist_ok=True)

target_path = out_dir / f"blend_{audio_acts[0]}.wav"
torchaudio.save(target_path, out, interface.sample_rate)

Audio(filename=Path(target_path))

In [ ]:
target_acts = [3 * i for i in [2, 5, 8, 10, 15, 18]]

scale = torchbend.BendingParameter(name="scale", value=1.0)
bias = torchbend.BendingParameter(name="bias", value=0.0)
cb = torchbend.Affine(scale=scale, bias=bias)

scales = [-5., -2., -0.1, 1., 0.1, 4., 10.]
biases = [-1., 0., 1.]
audio  = x[0][None]


out_dir = Path("outs/dac/encode_play")
os.makedirs(out_dir, exist_ok=True)
out_files = []

for t in target_acts: 
    interface.reset()
    interface.bend(cb, audio_acts[t], fn="encode")
    for s in scales:
        for b in biases:
            torch.cuda.empty_cache()
            scale.set_value(s)
            bias.set_value(b)
            z, _, _, _, _ = interface.encode(audio)
            out_bended = interface.decode(z).cpu()[0]

            out_path = out_dir / f"encode_{t}_s={s:.2f}_b={b:.2f}.wav"
            out_files.append((out_path, out_bended, interface.sample_rate))
            torchaudio.save(out_path, out_bended, interface.sample_rate)
            del out_bended, z
    
audio_grid(out_files)
        



GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Ou…

In [17]:
act_1.shape, act_2.shape

(torch.Size([1, 64, 176400]), torch.Size([1, 64, 176400]))

## Messing with the decoder


In [ ]:
interface.print_activations(fn="")

z0, _,_,_,_ = interface.encode(audio)

name,op,target,shape,args,kwargs
z,placeholder,z,"(1, 1024, s1)",(),{}
decoder_model_0_weight_v,get_attr,decoder.model.0.weight_v,"(1536, 1024, 7)",(),{}
decoder_model_0_weight_g,get_attr,decoder.model.0.weight_g,"(1536, 1, 1)",(),{}
_weight_norm_interface,call_function,aten._weight_norm_interface.default,None,"(decoder_model_0_weight_v, decoder_model_0_weight_g)",{}
getitem,call_function,<built-in function getitem>,"(1536, 1024, 7)","(_weight_norm_interface, 0)",{}
getitem_1,call_function,<built-in function getitem>,"(1536, 1, 1)","(_weight_norm_interface, 1)",{}
decoder_model_0_bias,get_attr,decoder.model.0.bias,"(1536,)",(),{}
convolution,call_function,aten.convolution.default,"(1, 1536, s1)","(z, getitem, decoder_model_0_bias, [1], [3], [1], False, [0], 1)",{}
view,call_function,aten.view.default,"(1, 1536, s1)","(convolution, [1, 1536, -1])",{}
decoder_model_1_block_0_alpha_1,get_attr,decoder.model.1.block.0.alpha,"(1, 1536, 1)",(),{}


'----------------------------------------  -------------  ----------------------------------------  ----------------  ------------------------------------------------------------------------------------------  --\nz                                         placeholder    z                                         (1, 1024, s1)     ()                                                                                          {}\ndecoder_model_0_weight_v                  get_attr       decoder.model.0.weight_v                  (1536, 1024, 7)   ()                                                                                          {}\ndecoder_model_0_weight_g                  get_attr       decoder.model.0.weight_g                  (1536, 1, 1)      ()                                                                                          {}\n_weight_norm_interface                    call_function  aten._weight_norm_interface.default                         (decoder_model_0_weight_v, dec

In [90]:
x = audio_files.as_list(channels=1, sr=interface.sample_rate, size=44100 * 4)
acts = interface.activations(r"?add_\d+", fn="decode")

audio_acts = []
for name, prop in acts.items(): 
    if len(prop.shape) < 1: continue
    if isinstance(prop.shape[-1], torch.SymInt):
        audio_acts.append(name)
print("number of valid activations : ", len(audio_acts))

z_0,_,_,_,_ = interface.encode(x[0][None])
z_1,_,_,_,_ = interface.encode(x[1][None])

number of valid activations :  41


In [ ]:
target_acts = [1, 3, 10, 20]

mask = torchbend.BendingParameter(name="prob", value=1.0)
cb = torchbend.ThresholdActivation(threshold=mask, dim=-2)

probs = [0.01, 0.1, 0.5, 0.8]
audio  = x[0][None]

out_dir = Path("outs/dac/decode_play")
os.makedirs(out_dir, exist_ok=True)
out_files = []

for t in target_acts: 
    interface.reset()
    interface.bend(cb, audio_acts[t], fn="decode")
    for p in probs:
        torch.cuda.empty_cache()
        mask.set_value(p)
        out_bended = interface.decode(z_0).cpu()[0]

        out_path = out_dir / f"decode_{t}_p={p:.2f}.wav"
        out_files.append((out_path, out_bended, interface.sample_rate))
        torchaudio.save(out_path, out_bended, interface.sample_rate)
        del out_bended
    
audio_grid(out_files)
        



GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Ou…

In [96]:
target_acts = [1, 3, 10, 20]

scale = torchbend.BendingParameter(name="scale", value=1.0)
bias = torchbend.BendingParameter(name="bias", value=0.0)
cb = torchbend.Affine(scale=scale, bias=bias)

scales = [-5., -2., -0.1, 1., 0.1, 4., 10.]
biases = [-1., 0., 1.]
audio  = x[0][None]


out_dir = Path("outs/dac/decode_play")
os.makedirs(out_dir, exist_ok=True)
out_files = []

for t in target_acts: 
    interface.reset()
    interface.bend(cb, audio_acts[t], fn="decode")
    for s in scales:
        for b in biases:
            torch.cuda.empty_cache()
            scale.set_value(s)
            bias.set_value(b)
            out_bended = interface.decode(z_0).cpu()[0]

            out_path = out_dir / f"decode_{t}_s={s:.2f}_b={b:.2f}.wav"
            out_files.append((out_path, out_bended, interface.sample_rate))
            torchaudio.save(out_path, out_bended, interface.sample_rate)
            del out_bended
    
audio_grid(out_files)
        



GridBox(children=(Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Output(), Ou…

In [158]:
from IPython.display import Audio
target_acts = [1, 3, 10, 20]

# interface.reset()

acts = interface.activations(r"?add_\d+", fn="decode")


z_0,_,_,_,_ = interface.encode(x[4][None])
z_1,_,_,_,_ = interface.encode(x[0][None])

z_0 = torch.nn.functional.interpolate(z_0, scale_factor=2.0, mode="linear")
z_1 = z_1.repeat_interleave(2, dim=-1)


def max_callback(act_1, act_2): 
    return torch.where(act_1 > act_2, act_1, act_2)

def min_callback(act_1, act_2):
    return torch.where(act_1 < act_2, act_1, act_2)

def sub_callback(act_1, act_2):
    # perm = torch.randperm(act_2.size(-2)).to(act_2.device)
    return (act_1 - act_2)/2

def stochastic_mix(z1, z2, prob=0.4, seed=654654):
    gen = torch.Generator(device=z1.device)
    if seed is not None:
        gen.manual_seed(seed)
    mask = torch.bernoulli(torch.full((z1.shape[-2],), prob, device=z1.device), generator=gen)
    mask = mask.reshape(1, -1, 1).to(z1)
    return z1 * mask + z2 * (1 - mask)

device = torch.device('cuda')
target_act = audio_acts[target_acts[1]]
act_1 = interface.get_activations(target_act, z=z_0, fn="decode")[target_act]
act_2 = interface.get_activations(target_act, z=z_1, fn="decode")[target_act]


act_mixed = {'z': z_0, target_act: max_callback(act_1, act_2)}
out = interface.from_activations(target_act, **act_mixed, fn="decode").cpu()[0]


out_dir = Path("outs/dac/dec_interp")
os.makedirs(out_dir, exist_ok=True)

target_path = out_dir / f"blend_{audio_acts[0]}.wav"
torchaudio.save(target_path, out, interface.sample_rate)

Audio(filename=Path(target_path))

In [180]:
interface.reset()
z_1,_,_,_,_ = interface.encode(x[1][None])

z_interp = torch.nn.functional.interpolate(z_1, scale_factor=1.8, mode="linear")

out = interface.decode(z_interp).cpu()[0]
target_path = "outs/dac/interp.wav"
torchaudio.save(target_path, out, interface.sample_rate)

Audio(filename=Path(target_path))